In [1]:
from dotenv import load_dotenv
load_dotenv()

True

#### Retriever

In [2]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [3]:
embedding_fn = HuggingFaceEmbeddings(model_name = "BAAI/bge-base-en-v1.5")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
vectorstore = Chroma(embedding_function = embedding_fn, persist_directory = '../../vectordb/intro_to_data_science')

In [5]:
retriever = vectorstore.as_retriever(search_type = 'mmr', search_kwargs = {'k': 3, 'lambda_mult': 0.6})

#### Chat Template

In [6]:
from langchain_core.prompts.chat import ChatPromptTemplate

In [7]:
chat_template = ChatPromptTemplate.from_messages([
    (
        'system',

        '''Answer the user question using the relevant documents provided. If the answer is not in the context, say you don't know.

        Also, mention the topic names the documents are taken from at the end of the response in the format:
        Sources: Source1, Source2, etc.'''
    ),

    (
        'human', 

        '''Question: {question}

        Relevant Documents:
        {context}'''
    )
])

#### Chat Model

In [8]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [9]:
chat = ChatGoogleGenerativeAI(
    model = 'gemini-3.1-flash-lite-preview',
    temperature = 0,
    max_output_tokens = 200,
    seed = 0
)

#### Chain

Context Injection:

- also called stuffing, inserts all retrieved docs into the prompt

Drawback:

- retrieved content may be large enough to not fit the model's context window limit
- also, the models tend to focus on the info at the beginning or the end of the retrieved doc list

`Document Refinement` can solve these problem but at the cost of more LLM calls

In [10]:
from IPython.display import Markdown
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [11]:
chain = {'question': RunnablePassthrough(), 'context': retriever} | chat_template | chat | StrOutputParser()

In [12]:
response = chain.invoke('What software do data scientists use?')

In [13]:
display(Markdown(response))

Data scientists use a variety of programming languages and software tools depending on their specific needs:

*   **Programming Languages:** R and Python are frequently used to create specific, ad-hoc tools for projects.
*   **Big Data Tools:** Software frameworks and databases designed for big data include Apache Hadoop, Apache Hbase, and MongoDB. Hadoop is particularly noted for distributing computational tasks across multiple computers.
*   **Business Intelligence & Visualization:** Power BI, SaS, Qlik, and Tableau are top-tier tools for business intelligence visualizations.
*   **Predictive Analytics & Research:** EViews is commonly used for econometric time-series models, while Stata is used for academic statistical and econometric research.

Sources: Programming Languages & Software Employed in Data Science - All the Tools You Need